# GPU Model Serving: Deploy PyTorch MLP to T4 Endpoint

This notebook deploys a PyTorch MLP model trained in `ray_gpu_model_training.ipynb` to a **GPU-powered model serving endpoint** (T4 instance) and makes example inference requests.

## Overview

1. **Load Model**: Retrieve a trained GPU model from Unity Catalog
2. **Create Endpoint**: Deploy to a model serving endpoint with T4 GPU
3. **Test Inference**: Make prediction requests to the endpoint

## Prerequisites

- **Cluster**: Run this notebook on the Serverless compute.
- Completed `ray_gpu_model_training.ipynb` with at least one registered model
- Unity Catalog access to the model registry
- Permissions to create model serving endpoints

## 1. Setup and Configuration

In [ ]:
# Core imports
import json
import time
import requests
import numpy as np
import pandas as pd

# Databricks SDK for model serving
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
    TrafficConfig,
    Route,
)

# MLflow for model info
import mlflow
from mlflow.tracking import MlflowClient

print("All imports successful!")

In [ ]:
# Configuration - Update these values to match your environment
catalog = "ryuta"
schema = "ray"

# Model to deploy (from ray_gpu_model_training.ipynb)
# Model IDs 90-99 are GPU models; we'll use model 90
MODEL_ID = 90
MODEL_NAME = f"{catalog}.{schema}.gpu_model_{MODEL_ID}_child"

# Endpoint configuration
ENDPOINT_NAME = f"gpu-model-{MODEL_ID}-serving"

# GPU workload configuration (T4 instance)
# Available GPU types: GPU_SMALL (T4), GPU_MEDIUM (A10G), GPU_LARGE (A100)
GPU_WORKLOAD_TYPE = ServingModelWorkloadType.GPU_SMALL  # T4 GPU
WORKLOAD_SIZE = "Small"  # Small, Medium, or Large
SCALE_TO_ZERO = True  # Enable scale to zero for cost savings

print(f"Configuration:")
print(f"  Model Name: {MODEL_NAME}")
print(f"  Endpoint Name: {ENDPOINT_NAME}")
print(f"  GPU Type: {GPU_WORKLOAD_TYPE.value} (NVIDIA T4)")
print(f"  Workload Size: {WORKLOAD_SIZE}")
print(f"  Scale to Zero: {SCALE_TO_ZERO}")

## 2. Verify Model in Unity Catalog

In [ ]:
# Set up MLflow to use Unity Catalog
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Get the latest version of the model
print(f"Looking up model: {MODEL_NAME}")

try:
    # List all versions of the model
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    
    if not versions:
        raise ValueError(f"No versions found for model {MODEL_NAME}")
    
    # Get the latest version
    latest_version = max(versions, key=lambda v: int(v.version))
    MODEL_VERSION = latest_version.version
    
    print(f"\nModel found!")
    print(f"  Name: {latest_version.name}")
    print(f"  Version: {MODEL_VERSION}")
    print(f"  Status: {latest_version.status}")
    print(f"  Run ID: {latest_version.run_id}")
    print(f"  Creation Time: {pd.Timestamp(latest_version.creation_timestamp, unit='ms')}")
    
except Exception as e:
    print(f"Error: {e}")
    print(f"\nMake sure you have run ray_gpu_model_training.ipynb first!")
    raise

In [ ]:
# Get model metrics from the training run
run = client.get_run(latest_version.run_id)

print("Model Training Metrics:")
for key, value in run.data.metrics.items():
    print(f"  {key}: {value:.4f}")

print("\nModel Parameters:")
for key, value in run.data.params.items():
    print(f"  {key}: {value}")

## 3. Create Model Serving Endpoint

Deploy the model to a GPU-powered serving endpoint using Databricks Model Serving with a T4 GPU instance.

In [ ]:
# Initialize Databricks Workspace Client
w = WorkspaceClient()

# Check if endpoint already exists
existing_endpoints = [ep.name for ep in w.serving_endpoints.list()]

if ENDPOINT_NAME in existing_endpoints:
    print(f"Endpoint '{ENDPOINT_NAME}' already exists.")
    print("Updating endpoint configuration...")
    
    # Update existing endpoint
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=MODEL_NAME,
                entity_version=MODEL_VERSION,
                workload_size=WORKLOAD_SIZE,
                workload_type=GPU_WORKLOAD_TYPE,
                scale_to_zero_enabled=SCALE_TO_ZERO,
            )
        ],
        traffic_config=TrafficConfig(
            routes=[Route(served_model_name=f"{MODEL_NAME}-{MODEL_VERSION}", traffic_percentage=100)]
        ),
    )
    print(f"Endpoint '{ENDPOINT_NAME}' updated successfully!")
else:
    print(f"Creating new endpoint '{ENDPOINT_NAME}'...")
    print(f"  Model: {MODEL_NAME} (version {MODEL_VERSION})")
    print(f"  GPU: {GPU_WORKLOAD_TYPE} (T4)")
    print(f"  Size: {WORKLOAD_SIZE}")
    print("\nThis may take several minutes...")
    
    # Create new endpoint
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version=MODEL_VERSION,
                    workload_size=WORKLOAD_SIZE,
                    workload_type=GPU_WORKLOAD_TYPE,
                    scale_to_zero_enabled=SCALE_TO_ZERO,
                )
            ],
        ),
    )
    print(f"\nEndpoint '{ENDPOINT_NAME}' created successfully!")

In [ ]:
# Get endpoint status and URL
endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)

print(f"Endpoint Status:")
print(f"  Name: {endpoint.name}")
print(f"  State: {endpoint.state.ready}")
print(f"  Config Update: {endpoint.state.config_update}")

# Get the serving URL
SERVING_URL = f"{w.config.host}/serving-endpoints/{ENDPOINT_NAME}/invocations"
print(f"\nServing URL: {SERVING_URL}")

## 4. Load Test Data

Load sample data from the same table used for training to make inference requests.

In [ ]:
# Load test data from the same table used in training
TABLE_NAME = f"{catalog}.{schema}.synthetic_data"

print(f"Loading sample data from {TABLE_NAME}...")
df_spark = spark.table(TABLE_NAME).limit(100)
df = df_spark.toPandas()

# Get feature columns (same as training)
feature_columns = [col for col in df.columns if col.startswith('feature_')]
X_sample = df[feature_columns].values
y_actual = df['label'].values

print(f"Loaded {len(df)} samples with {len(feature_columns)} features")
print(f"Feature columns: {feature_columns[:5]}... (showing first 5)")
print(f"Sample shape: {X_sample.shape}")

In [ ]:
# Get the feature indices used by this specific model
# The model was trained on a subset of features

# Download feature_indices.json artifact from the model run
artifact_path = client.download_artifacts(latest_version.run_id, "feature_indices.json")

with open(artifact_path, 'r') as f:
    feature_indices = json.load(f)

print(f"Model was trained on {len(feature_indices)} features")
print(f"Feature indices: {feature_indices[:10]}... (showing first 10)")

# Select only the features used by the model
X_model_input = X_sample[:, feature_indices]
print(f"\nModel input shape: {X_model_input.shape}")

## 5. Make Inference Requests

Send prediction requests to the deployed model serving endpoint.

In [ ]:
def get_databricks_token():
    """Get Databricks API token for authentication"""
    # In a Databricks notebook, we can get the token from the context
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    return ctx.apiToken().getOrElse(None)

def query_endpoint(endpoint_name, input_data, token):
    """
    Query the model serving endpoint with input data.
    
    Args:
        endpoint_name: Name of the serving endpoint
        input_data: numpy array or list of input features
        token: Databricks API token
    
    Returns:
        Prediction results from the endpoint
    """
    url = f"{w.config.host}/serving-endpoints/{endpoint_name}/invocations"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    
    # Format input data for the endpoint
    # PyTorch models expect dataframe-split format
    if isinstance(input_data, np.ndarray):
        input_data = input_data.tolist()
    
    payload = {
        "dataframe_split": {
            "data": input_data
        }
    }
    
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code != 200:
        raise Exception(f"Request failed: {response.status_code} - {response.text}")
    
    return response.json()

# Get authentication token
TOKEN = get_databricks_token()
print("Authentication token retrieved successfully!")

In [ ]:
# Make a single prediction request (first sample)
print("Making a single prediction request...")
print(f"Input shape: {X_model_input[0:1].shape}")

start_time = time.time()
result = query_endpoint(ENDPOINT_NAME, X_model_input[0:1], TOKEN)
latency = (time.time() - start_time) * 1000

print(f"\nPrediction Result:")
print(f"  Raw output: {result}")
print(f"  Latency: {latency:.2f} ms")

# Parse prediction
prediction_proba = result['predictions'][0]
if isinstance(prediction_proba, list):
    prediction_proba = prediction_proba[0]
prediction = 1 if prediction_proba > 0.5 else 0

print(f"\nInterpreted Result:")
print(f"  Probability: {prediction_proba:.4f}")
print(f"  Predicted Class: {prediction}")
print(f"  Actual Class: {y_actual[0]}")
print(f"  Correct: {'Yes' if prediction == y_actual[0] else 'No'}")

In [ ]:
# Make batch prediction request (first 10 samples)
BATCH_SIZE = 10

print(f"Making batch prediction request ({BATCH_SIZE} samples)...")
print(f"Input shape: {X_model_input[:BATCH_SIZE].shape}")

start_time = time.time()
batch_result = query_endpoint(ENDPOINT_NAME, X_model_input[:BATCH_SIZE], TOKEN)
batch_latency = (time.time() - start_time) * 1000

print(f"\nBatch Prediction Results:")
print(f"  Total latency: {batch_latency:.2f} ms")
print(f"  Per-sample latency: {batch_latency/BATCH_SIZE:.2f} ms")

# Parse predictions
predictions_proba = batch_result['predictions']
if isinstance(predictions_proba[0], list):
    predictions_proba = [p[0] for p in predictions_proba]
predictions = [1 if p > 0.5 else 0 for p in predictions_proba]

# Compare with actual labels
print(f"\nPrediction Comparison:")
print(f"{'Sample':<8} {'Probability':<12} {'Predicted':<10} {'Actual':<8} {'Correct':<8}")
print("-" * 50)

correct = 0
for i in range(BATCH_SIZE):
    is_correct = predictions[i] == y_actual[i]
    correct += is_correct
    print(f"{i:<8} {predictions_proba[i]:<12.4f} {predictions[i]:<10} {y_actual[i]:<8} {'Yes' if is_correct else 'No':<8}")

print(f"\nBatch Accuracy: {correct}/{BATCH_SIZE} ({100*correct/BATCH_SIZE:.1f}%)")

In [ ]:
# Performance benchmark: Multiple requests to measure throughput
print("Running performance benchmark...")

NUM_REQUESTS = 20
SAMPLES_PER_REQUEST = 10

latencies = []
total_samples = 0

for i in range(NUM_REQUESTS):
    # Cycle through test samples
    start_idx = (i * SAMPLES_PER_REQUEST) % len(X_model_input)
    end_idx = min(start_idx + SAMPLES_PER_REQUEST, len(X_model_input))
    batch = X_model_input[start_idx:end_idx]
    
    start_time = time.time()
    _ = query_endpoint(ENDPOINT_NAME, batch, TOKEN)
    latency = (time.time() - start_time) * 1000
    
    latencies.append(latency)
    total_samples += len(batch)
    
    if (i + 1) % 5 == 0:
        print(f"  Completed {i + 1}/{NUM_REQUESTS} requests")

# Calculate statistics
latencies = np.array(latencies)
total_time = sum(latencies)

print(f"\nPerformance Benchmark Results:")
print(f"  Total requests: {NUM_REQUESTS}")
print(f"  Total samples: {total_samples}")
print(f"  Samples per request: {SAMPLES_PER_REQUEST}")
print(f"\nLatency Statistics (ms):")
print(f"  Mean: {latencies.mean():.2f}")
print(f"  Std: {latencies.std():.2f}")
print(f"  Min: {latencies.min():.2f}")
print(f"  Max: {latencies.max():.2f}")
print(f"  P50: {np.percentile(latencies, 50):.2f}")
print(f"  P95: {np.percentile(latencies, 95):.2f}")
print(f"  P99: {np.percentile(latencies, 99):.2f}")
print(f"\nThroughput:")
print(f"  Requests/second: {NUM_REQUESTS / (total_time / 1000):.2f}")
print(f"  Samples/second: {total_samples / (total_time / 1000):.2f}")

## 6. Cleanup (Optional)

Delete the serving endpoint to stop incurring costs. Uncomment the cell below to delete.

In [ ]:
# Uncomment to delete the endpoint
# WARNING: This will permanently delete the serving endpoint

# print(f"Deleting endpoint '{ENDPOINT_NAME}'...")
# w.serving_endpoints.delete(name=ENDPOINT_NAME)
# print("Endpoint deleted successfully!")

## Summary

This notebook demonstrated how to:

1. **Load a trained model** from Unity Catalog (model trained in `ray_gpu_model_training.ipynb`)
2. **Deploy to GPU endpoint** using Databricks Model Serving with T4 GPU (`GPU_SMALL` workload type)
3. **Make inference requests** using the REST API with proper authentication
4. **Benchmark performance** measuring latency and throughput

### Key Configuration Options

| Parameter | Value | Description |
|-----------|-------|-------------|
| `GPU_WORKLOAD_TYPE` | `GPU_SMALL` | NVIDIA T4 GPU |
| `WORKLOAD_SIZE` | `Small` | Endpoint compute size |
| `SCALE_TO_ZERO` | `True` | Enable scale-to-zero for cost savings |

### GPU Workload Types

| Type | GPU | Use Case |
|------|-----|----------|
| `GPU_SMALL` | NVIDIA T4 | Cost-effective inference |
| `GPU_MEDIUM` | NVIDIA A10G | Balanced performance |
| `GPU_LARGE` | NVIDIA A100 | High-throughput inference |

### Next Steps

- Monitor endpoint metrics in the Databricks UI
- Set up autoscaling for production workloads
- Implement A/B testing with traffic splitting
- Add custom preprocessing with model wrappers